In [1]:
!pip install pyodbc sqlalchemy

In [2]:
import pyodbc
print(pyodbc.drivers())


['SQL Server', 'SQL Server Native Client 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'ODBC Driver 18 for SQL Server']


In [4]:
from sqlalchemy import create_engine
import pandas as pd
import urllib

server = "192.168.3.122"
database = "DBMGERP"

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Test Query
df = pd.read_sql("select * from AccChequeBookDetail", engine)
df.head()


OperationalError: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 18 for SQL Server]Named Pipes Provider: Could not open a connection to SQL Server [5].  (5) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 18 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 18 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to 192.168.3.122. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (5)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [5]:
from sqlalchemy import create_engine
import pandas as pd
import urllib

server = "192.168.3.122"
database = "DBMGERP"

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server},1433;"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

df = pd.read_sql("SELECT TOP 5 * FROM AccChequeBookDetail", engine)
df.head()


OperationalError: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 18 for SQL Server]TCP Provider: No connection could be made because the target machine actively refused it.\r\n (10061) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 18 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 18 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to 192.168.3.122,1433. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (10061)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [11]:
from sqlalchemy import create_engine
import pandas as pd
import urllib

# ---------------------------
# Database connection setup
# ---------------------------
server = "localhost"
database = "DBMGERP"

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "Encrypt=no;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# ---------------------------
# Stored Procedure parameters
# ---------------------------
report_type_id = 0
fiscal_year_id = 0
from_date = '2025-01-01'
to_date = '2025-12-31'
company_group_id = 1
is_realize = 0

# ---------------------------
# SQL Query to execute procedure
# ---------------------------
sql_query = f"""
EXEC Report_ExportInvoiceSummary
    @ReportTypeID={report_type_id},
    @FiscalYearID={fiscal_year_id},
    @FromDate='{from_date}',
    @ToDate='{to_date}',
    @CompanyGroupID={company_group_id},
    @isRealize={is_realize}
"""

# ---------------------------
# Execute query and load into DataFrame
# ---------------------------
try:
    df = pd.read_sql(sql_query, engine)
    print("Query executed successfully! Previewing first 10 rows:")
    print(df.head(10))
except Exception as e:
    print("Error executing stored procedure:", e)


Error executing stored procedure: This result object does not return rows. It has been closed automatically.


In [ ]:
pd.read_sql('Report_ExportInvoiceSummary @ReportTypeID=0,@FiscalYearID=0,@FromDate='2025-01-01',@ToDate='2025-12-31',@CompanyGroupID=1,@isRealize=0,@SalesInvoiceID=41753")

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
### load EIS_WO and EIS_SW Dataset 
EIS_WO = pd.read_csv("EIS_WO.csv")
EIS_SW = pd.read_csv("EIS_SW.csv")

In [3]:
EIS_WO.head(5)

,LC Factory,LC/SC No,LC/SC Value,LC/SC Quantity,LC/SC Payment Term,Invoice No,Invoice Date,EXP No,EXP Date,Buyer,...,Bank Submit Date,Expected Realize Date,Is FDBC,Bank Name,Shipping Mode,Invoice Status,Realize NO,Realize Date,Realize Value,Is Realize
0,MGSL,MGSL/Polly Slim Fit Shirt S. 8,96325.68,20677,EOM 63,2310013,09-May-2023,042/006812/23,27-May-2023,H&M,...,08-Jun-2023,NaN,Yes,ABL,Sea,ExFactory Pending,DR-06-23-002,08-Jun-2023,1706.60,Yes
1,MGSL,MGSL/Polly Slim Fit Shirt S. 8,96325.68,20677,EOM 63,2310015,09-May-2023,042/006815/23,27-May-2023,H&M,...,08-Jun-2023,NaN,Yes,ABL,Sea,ExFactory Pending,DR-06-23-002,08-Jun-2023,87.40,Yes
2,MGL,MG-Pepe Jeans Denim-24,1536453.18,118136,90 Days,2310017,10-May-2023,1949/009162/23,12-Apr-2023,PEPE JEANS SL,...,15-May-2023,NaN,Yes,EXIM,Sea,ExFactory Pending,DR-09-23-005,04-Sep-2023,6268.22,Yes
3,MGSL,MGSL/Robin Oxford Shirt S. 8,2622975.35,501569,60 Days,2310019,11-May-2023,042/006782/23,25-May-2023,H&M,...,NaN,NaN,No,ABL,Sea,ExFactory Pending,NaN,NaN,NaN,No
4,MGSL,MGSL/Robin Oxford Shirt S. 8,2622975.35,501569,60 Days,2310020,05-May-2023,042/006783/23,25-May-2023,H&M,...,08-Aug-2023,NaN,Yes,ABL,Sea,ExFactory Pending,DR-08-23-015,08-Aug-2023,2823.10,Yes


In [4]:
EIS_WO['Category'] = 'Woven'
EIS_SW['Category'] = 'Sweater'

In [5]:
EIS_WO.columns

Index(['LC Factory', 'LC/SC No', 'LC/SC Value', 'LC/SC Quantity',
       'LC/SC Payment Term', 'Invoice No', 'Invoice Date', 'EXP No',
       'EXP Date', 'Buyer', 'PO No', 'Merchandiser', 'Destination No',
       'Invoice Qty', 'Carton Qty', 'Invoice Value', 'Discount', 'Commission',
       'ExfactoryDate', 'Ex Factory No', 'Ex Factory', 'On Board Date',
       'Payment Due Date', 'OnBoardDiff', 'Feeder Vessel No',
       'Mother Vessel No', 'Shipping Bill No', 'Shipping Bill Date', 'BL No',
       'BL Date', 'FCR No', 'FCR Date', 'Bank Ref No', 'Bank Submit Date',
       'Expected Realize Date', 'Is FDBC', 'Bank Name', 'Shipping Mode',
       'Invoice Status', 'Realize NO', 'Realize Date', 'Realize Value',
       'Is Realize', 'Category'],
      dtype='object')

In [6]:
print("Total Category Count in Woven : ", EIS_WO['Category'].count())
print("Total Category Count in Sweater: ", EIS_SW['Category'].count())

Total Category Count in Woven :  42168
Total Category Count in Sweater:  1428


In [7]:
# ### Concat EIS_WO + EIS_SW = EIS DataFrame

# EIS = pd.concat([EIS_WO,EIS_SW],ignore_index=True)
# EIS.Sample(10)


# Concat EIS_WO + EIS_SW = EIS DataFrame
EIS = pd.concat([EIS_WO, EIS_SW], ignore_index=True)

# Show random 10 rows
EIS.sample(10)


,LC Factory,LC/SC No,LC/SC Value,LC/SC Quantity,LC/SC Payment Term,Invoice No,Invoice Date,EXP No,EXP Date,Buyer,...,Expected Realize Date,Is FDBC,Bank Name,Shipping Mode,Invoice Status,Realize NO,Realize Date,Realize Value,Is Realize,Category
22825,MGSL,MGSL/H&M-MENS-S.0,7963192.82,1672612,EOM+63,2414929,09-Oct-2024,00000042-018926-2024,09-Oct-2024,H&M,...,NaN,Yes,ABL,Sea,Incentive Pending,DR-01-25-042,15-Jan-2025,222.36,Yes,Woven
34870,MGSL,MGSL/H&M Kids S2,7104090.79,2068406,EOM 63,2508006,19-Jun-2025,00001689-017574-2025,19-Jun-2025,H&M,...,NaN,Yes,DBBL,Sea,Incentive Pending,DR-09-25-032,09-Sep-2025,3490.06,Yes,Woven
35864,MGSL,MGSL/H&M Kids S2,7104090.79,2068406,EOM 63,2509075,16-Jul-2025,00001689-020343-2025,17-Jul-2025,H&M,...,NaN,Yes,DBBL,Sea,Incentive Pending,DR-11-25-063,30-Nov-2025,7249.08,Yes,Woven
40754,MGSL,MGSL/H&M Kids S2,7104090.79,2068406,EOM 63,2600227,04-Jan-2026,00001689-000411-2026,04-Jan-2026,H&M,...,NaN,No,DBBL,Sea,Bank Submit Pending,NaN,NaN,NaN,No,Woven
34080,MGSL,MGSL/H&M Kids S2,7104090.79,2068406,EOM 63,2507183,25-May-2025,00001689-015845-2025,25-May-2025,H&M,...,NaN,Yes,DBBL,Sea,Incentive Pending,DR-08-25-057,26-Aug-2025,5299.56,Yes,Woven
34153,MGSL,MGSL/H&M-Mens DBL-S.02,12435079.87,3145397,EOM 63,2507256,25-May-2025,00001689-015917-2025,25-May-2025,H&M,...,NaN,Yes,DBBL,Sea,Incentive Pending,DR-08-25-058,27-Aug-2025,3680.00,Yes,Woven
3020,MGSL,MGSL/H&M-MENS -S.07,2948973.12,763766,EOM 63,2313161,10-Sep-2023,042/013644/23,10-Sep-2023,H&M,...,NaN,Yes,ABL,Sea,Incentive Pending,DR-10-23-036,05-Oct-2023,30.30,Yes,Woven
12221,MGL,CESITF2300040,538253.00,66900,90 Days,2403680,04-Mar-2024,0947/000751/2024,05-Mar-2024,EL CORTE INGLES,...,NaN,Yes,NBL,Sea,Realize Pending,NaN,NaN,NaN,No,Woven
22089,MGSL,MGSL/H&M Kids DBBL-S.0,7767885.39,2760068,EOM+63,2414128,26-Sep-2024,00001689-017851-2024,26-Sep-2024,H&M,...,NaN,Yes,DBBL,Sea,Incentive Pending,DR-12-24-008,03-Dec-2024,180.88,Yes,Woven
3375,MGSL,MGSL/H&M Shirt S.08,4509235.10,950000,EOM+63,2313548,21-Sep-2023,042/014288/23,21-Sep-2023,H&M,...,NaN,No,ABL,Sea,Bank Submit Pending,NaN,NaN,NaN,No,Woven


In [8]:
EIS.shape

(43596, 44)

In [26]:
### Random shuffle the data

EIS = EIS.sample(frac=1).reset_index(drop=True)

In [27]:
EIS.head(5)

,Company,LCSC_No,LCSC_Value,LCSC_Qty,LCSC_Payment_Term,Invoice_No,Invoice_Date,EXP_No,EXP_Date,Buyer,...,Is_FDBC?,Bank,Shipping_Mode,Status,Realize_No,Realize_Date,Realize_Value,Is_Realize?,Category,Payment_After_Ex_Factory
0,MGSL,MGSL/H&M-Mens DBL-S.0,11664021.68,2755553,EOM+63,2413785,2024-09-19,00001689-017246-2024,2024-09-19,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-11-24-003,2024-10-31,514.48,Yes,Woven,6.0
1,MGSL,MGSL/H&M Shirt S.08,4509235.10,950000,EOM+63,2312650,2023-08-27,042/012833/23,2023-08-27,H&M,...,Yes,ABL,Sea,Incentive Pending,DR-09-23-015,2023-09-23,201.30,Yes,Woven,6.0
2,MGSL,MGSL/H&M Kids DBBL-S.0,7767885.39,2760068,EOM+63,2417645,2024-12-05,00001689-025383-2024,2024-12-05,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-03-25-016,2025-03-06,343.20,Yes,Woven,6.0
3,MGSL,MGSL/Women Shirt S.2,2995246.39,619756,EOM+63,2504335,2025-03-18,00001689-009552-2025,2025-03-18,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-05-25-032,2025-05-20,945.45,Yes,Woven,6.0
4,MGSL,MGSL/H&M-MENS-S.0,7963192.82,1672612,EOM+63,2403814,2024-03-06,042/006626/24,2024-03-06,H&M,...,Yes,ABL,Sea,Incentive Pending,DR-03-24-031,2024-03-24,827.17,Yes,Woven,6.0


In [11]:
EIS['Category'].value_counts()


Category
Woven      42168
Sweater     1428
Name: count, dtype: int64

In [12]:
EIS.shape

(43596, 44)

In [13]:
payment_chart_DF = pd.read_csv('Buyer_Payment_Chart.csv')
payment_chart_DF.head(5)

,Sl. No.,Buyer,Export Responsible,Ex-Factory,Days,Onboard,Days.1,B/L Collection,Days.2,Bank Doc Submission,Days.3,Payment Term,Payment Date,Remarks,Payment After Ex-Factory
0,1,Assurer Worldwide,NaN,01-01-2026,7.0,08-01-2026,3.0,11-01-2026,NaN,NaN,NaN,10,21-01-2026,not update,20
1,2,Assurer Worldwide Apparel LLC.,NaN,01-01-2026,7.0,08-01-2026,3.0,11-01-2026,NaN,NaN,NaN,10,21-01-2026,not update,20
2,3,Assurer Worldwide Apparel LLC-FZ,NaN,01-01-2026,7.0,08-01-2026,3.0,11-01-2026,NaN,NaN,NaN,10,21-01-2026,not update,20
3,4,Assurer Worldwide Apparel LLC-FZ,NaN,01-01-2026,7.0,08-01-2026,3.0,11-01-2026,NaN,NaN,NaN,10,21-01-2026,not update,20
4,5,Assurer Worldwide Apparel LLC-FZ,NaN,01-01-2026,7.0,08-01-2026,3.0,11-01-2026,NaN,NaN,NaN,10,21-01-2026,not update,20


In [14]:
EIS = EIS.merge(
    payment_chart_DF[['Buyer', 'Payment After Ex-Factory']],
    on = 'Buyer',
    how ='left'
)

EIS.head()

,LC Factory,LC/SC No,LC/SC Value,LC/SC Quantity,LC/SC Payment Term,Invoice No,Invoice Date,EXP No,EXP Date,Buyer,...,Is FDBC,Bank Name,Shipping Mode,Invoice Status,Realize NO,Realize Date,Realize Value,Is Realize,Category,Payment After Ex-Factory
0,MGSL,MGSL/H&M-Mens DBL-S.02,12435079.87,3145397,EOM 63,2511040,18-Sep-2025,00001689-024913-2025,18-Sep-2025,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-01-26-005,04-Jan-2026,215.22,Yes,Woven,6.0
1,MGSL,MGSL/H&M Kids S2,7104090.79,2068406,EOM 63,2601605,01-Feb-2026,00001689-003345-2026,01-Feb-2026,H&M,...,No,DBBL,Sea,On Board Pending,NaN,NaN,NaN,No,Woven,6.0
2,MGSL,MGSL/H&M Kids S2,7104090.79,2068406,EOM 63,2510585,28-Aug-2025,00001689-023682-2025,28-Aug-2025,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-12-25-050,22-Dec-2025,301.78,Yes,Woven,6.0
3,MGSL,MGSL/H&M Kids DBBL-S.0,7767885.39,2760068,EOM+63,2414253,29-Sep-2024,00001689-018149-2024,29-Sep-2024,H&M,...,Yes,DBBL,Sea/Air,Incentive Pending,DR-12-24-035,18-Dec-2024,5090.11,Yes,Woven,6.0
4,MGSL,MGSL/H&M Kids DBBL-S.0,7767885.39,2760068,EOM+63,2504713,23-Mar-2025,00001689-010303-2025,23-Mar-2025,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-05-25-037,21-May-2025,49.59,Yes,Woven,6.0


In [15]:
EIS.shape

(43596, 45)

In [16]:
EIS.columns

Index(['LC Factory', 'LC/SC No', 'LC/SC Value', 'LC/SC Quantity',
       'LC/SC Payment Term', 'Invoice No', 'Invoice Date', 'EXP No',
       'EXP Date', 'Buyer', 'PO No', 'Merchandiser', 'Destination No',
       'Invoice Qty', 'Carton Qty', 'Invoice Value', 'Discount', 'Commission',
       'ExfactoryDate', 'Ex Factory No', 'Ex Factory', 'On Board Date',
       'Payment Due Date', 'OnBoardDiff', 'Feeder Vessel No',
       'Mother Vessel No', 'Shipping Bill No', 'Shipping Bill Date', 'BL No',
       'BL Date', 'FCR No', 'FCR Date', 'Bank Ref No', 'Bank Submit Date',
       'Expected Realize Date', 'Is FDBC', 'Bank Name', 'Shipping Mode',
       'Invoice Status', 'Realize NO', 'Realize Date', 'Realize Value',
       'Is Realize', 'Category', 'Payment After Ex-Factory'],
      dtype='object')

In [17]:
# Drop unnecessary columns from EIS
columns_to_drop = [
    'PO No',
    'Discount',
    'Commission',
    'Feeder Vessel No',
    'Mother Vessel No',
    'FCR No',
    'FCR Date',
    'Expected Realize Date'
]

EIS.drop(columns=columns_to_drop, inplace=True)

# Optional: check remaining columns
print("Remaining columns:", EIS.columns.tolist())


Remaining columns: ['LC Factory', 'LC/SC No', 'LC/SC Value', 'LC/SC Quantity', 'LC/SC Payment Term', 'Invoice No', 'Invoice Date', 'EXP No', 'EXP Date', 'Buyer', 'Merchandiser', 'Destination No', 'Invoice Qty', 'Carton Qty', 'Invoice Value', 'ExfactoryDate', 'Ex Factory No', 'Ex Factory', 'On Board Date', 'Payment Due Date', 'OnBoardDiff', 'Shipping Bill No', 'Shipping Bill Date', 'BL No', 'BL Date', 'Bank Ref No', 'Bank Submit Date', 'Is FDBC', 'Bank Name', 'Shipping Mode', 'Invoice Status', 'Realize NO', 'Realize Date', 'Realize Value', 'Is Realize', 'Category', 'Payment After Ex-Factory']


In [18]:
# Define rename mapping
rename_mapping = {
    'LC Factory': "Company",
    'LC/SC No': "LCSC_No",
    'LC/SC Value': "LCSC_Value",
    'LC/SC Quantity': "LCSC_Qty",
    'LC/SC Payment Term': "LCSC_Payment_Term",
    'Invoice No': "Invoice_No",
    'Invoice Date': "Invoice_Date",
    'EXP No': "EXP_No",
    'EXP Date': "EXP_Date",
    'Destination No': "Destination",
    'Invoice Qty': "Invoice_Qty",
    'Carton Qty': "Carton_Qty",
    'Invoice Value': "Invoice_Value",
    'ExfactoryDate': "Ex_Factory_Date",
    'Ex Factory No': "Ex_Factory_No",
    'Ex Factory': "Ex_Factory",
    'On Board Date': "On_Board_Date",
    'Payment Due Date': "Payment_Due_Date",
    'OnBoardDiff': "On_Board_Diff",
    'Shipping Bill No': "Shipping_Bill_No",
    'Shipping Bill Date': "Shipping_Bill_Date",
    'BL No': "BL_No",
    'BL Date': "BL_Date",
    'Bank Ref No': "FDBC_No",
    'Bank Submit Date': "Bank_Submit_Date",
    'Expected Realize Date': "Expected_Realize_Date",
    'Is FDBC': "Is_FDBC?",
    'Bank Name': "Bank",
    'Shipping Mode': "Shipping_Mode",
    'Invoice Status': "Status",
    'Realize NO': "Realize_No",
    'Realize Date': "Realize_Date",
    'Realize Value': "Realize_Value",
    'Is Realize': "Is_Realize?",
    'Payment After Ex-Factory': "Payment_After_Ex_Factory"
}

# Rename columns safely, ignoring missing ones
existing_cols = {k: v for k, v in rename_mapping.items() if k in EIS.columns}
EIS.rename(columns=existing_cols, inplace=True)

# Optional: check updated columns
print("Columns after rename:", EIS.columns.tolist())


Columns after rename: ['Company', 'LCSC_No', 'LCSC_Value', 'LCSC_Qty', 'LCSC_Payment_Term', 'Invoice_No', 'Invoice_Date', 'EXP_No', 'EXP_Date', 'Buyer', 'Merchandiser', 'Destination', 'Invoice_Qty', 'Carton_Qty', 'Invoice_Value', 'Ex_Factory_Date', 'Ex_Factory_No', 'Ex_Factory', 'On_Board_Date', 'Payment_Due_Date', 'On_Board_Diff', 'Shipping_Bill_No', 'Shipping_Bill_Date', 'BL_No', 'BL_Date', 'FDBC_No', 'Bank_Submit_Date', 'Is_FDBC?', 'Bank', 'Shipping_Mode', 'Status', 'Realize_No', 'Realize_Date', 'Realize_Value', 'Is_Realize?', 'Category', 'Payment_After_Ex_Factory']


In [19]:
EIS.sample(10)

,Company,LCSC_No,LCSC_Value,LCSC_Qty,LCSC_Payment_Term,Invoice_No,Invoice_Date,EXP_No,EXP_Date,Buyer,...,Is_FDBC?,Bank,Shipping_Mode,Status,Realize_No,Realize_Date,Realize_Value,Is_Realize?,Category,Payment_After_Ex_Factory
34729,MGSL,MGSL/H&M Boys S9,12327213.47,4104494,EOM+63,2401517,27-Jan-2024,042/002660/24,27-Jan-2024,H&M,...,Yes,ABL,Sea,Incentive Pending,DR-03-24-037,06-Mar-2024,59.75,Yes,Woven,6.0
31892,MGSL,MGSL/H&M Kids DBBL-S.0,7767885.39,2760068,EOM+63,2412561,26-Aug-2024,00001689-015203-2024,26-Aug-2024,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-09-24-018,10-Sep-2024,1771.56,Yes,Woven,6.0
11522,MGSL,MGSL/H&M-Mens DBL-S.02,12435079.87,3145397,EOM 63,2512387,13-Nov-2025,00001689-029001-2025,13-Nov-2025,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-01-26-042,18-Jan-2026,134.78,Yes,Woven,6.0
39574,MGSL,MGSL/H&M-Mens DBL-S.0,11664021.68,2755553,EOM+63,2504487,20-Mar-2025,00001689-010146-2025,20-Mar-2025,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-05-25-030,08-May-2025,1200.50,Yes,Woven,6.0
545,MGSL,MGSL/H&M Girls S0,1504550.35,367402,EOM+63,2411014,25-Jul-2024,00000042-016674-2024,25-Jul-2024,H&M,...,Yes,ABL,Sea,Incentive Pending,DR-08-24-007,11-Aug-2024,335.71,Yes,Woven,6.0
200,MGSL,MGSL/H&M-Mens DBL-S.02,12435079.87,3145397,EOM 63,2513525,14-Dec-2025,00001689-031932-2025,14-Dec-2025,H&M,...,No,DBBL,Sea,Bank Submit Pending,NaN,NaN,NaN,No,Woven,6.0
24825,MGSL,MGSL/H&M Kids S2,7104090.79,2068406,EOM 63,2508450,03-Jul-2025,00001689-018946-2025,03-Jul-2025,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-11-25-027,11-Nov-2025,181.45,Yes,Woven,6.0
13095,MGSL,MGSL/PRINCETON SHIRT-8757 S.09,911882.48,283036,EOM+63,2401382,24-Jan-2024,042/02325/24,25-Jan-2024,H&M,...,Yes,ABL,Sea,Incentive Pending,DR-05-24-049,08-May-2024,8.10,Yes,Woven,6.0
261,MGSL,MGSL/H&M-Mens DBL-S.0,11664021.68,2755553,EOM+63,2418433,15-Dec-2024,00001689-026697-2024,15-Dec-2024,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-02-25-009,04-Feb-2025,1270.08,Yes,Woven,6.0
33654,MGNSL,MGNSL/VS-NEXT/002,625217.27,167454,At Sight,2601460,28-Jan-2026,000042-000501-2026,28-Jan-2026,NEXT,...,No,ABL,Sea/Air,On Board Pending,NaN,NaN,NaN,No,Woven,40.0


In [20]:
EIS.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43596 entries, 0 to 43595
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Company                   43596 non-null  object 
 1   LCSC_No                   43596 non-null  object 
 2   LCSC_Value                43596 non-null  float64
 3   LCSC_Qty                  43596 non-null  int64  
 4   LCSC_Payment_Term         43593 non-null  object 
 5   Invoice_No                43596 non-null  object 
 6   Invoice_Date              43596 non-null  object 
 7   EXP_No                    43556 non-null  object 
 8   EXP_Date                  43596 non-null  object 
 9   Buyer                     43596 non-null  object 
 10  Merchandiser              43596 non-null  object 
 11  Destination               43593 non-null  object 
 12  Invoice_Qty               43596 non-null  int64  
 13  Carton_Qty                42493 non-null  float64
 14  Invoic

In [21]:
EIS.describe()

,LCSC_Value,LCSC_Qty,Invoice_Qty,Carton_Qty,Invoice_Value,On_Board_Diff,Realize_Value,Payment_After_Ex_Factory
count,4.359600e+04,4.359600e+04,43596.000000,42493.000000,43596.000000,41592.000000,40118.000000,43305.000000
mean,6.161161e+06,1.658518e+06,970.897169,35.820770,4193.904099,7.891662,4095.952205,10.627018
std,4.289149e+06,1.331716e+06,2940.803135,191.602905,13790.671193,18.707503,13378.661230,17.488623
min,7.627500e+02,0.000000e+00,2.000000,0.000000,5.700000,-373.000000,7.470000,6.000000
25%,2.622975e+06,5.015690e+05,61.000000,1.000000,241.800000,6.000000,239.010000,6.000000
50%,4.542005e+06,1.244813e+06,174.000000,3.000000,656.035000,8.000000,631.750000,6.000000
75%,1.166402e+07,2.755553e+06,660.000000,11.000000,2489.940000,10.000000,2373.340000,6.000000
max,1.347979e+07,9.527500e+06,99066.000000,7711.000000,480570.440000,430.000000,480570.220000,120.000000


In [22]:
### Change Data Type as column wise

# List of columns by desired type
text_cols = [
    'Company', 'LCSC_No', 'LCSC_Payment_Term', 'Invoice_No', 'EXP_No', 'Buyer',
    'Merchandiser', 'Destination', 'Ex_Factory_No', 'Ex_Factory',
    'Shipping_Bill_No', 'BL_No', 'FDBC_No', 'Is_FDBC?', 'Bank',
    'Shipping_Mode', 'Status', 'Realize_No', 'Is_Realize?', 'Category'
]

date_cols = [
    'Invoice_Date', 'EXP_Date', 'Ex_Factory_Date', 'On_Board_Date', 
    'Payment_Due_Date', 'Shipping_Bill_Date', 'Realize_Date'
]

float_cols = [
    'LCSC_Value', 'Invoice_Value', 'On_Board_Diff', 'Realize_Value', 
    'Payment_After_Ex_Factory'
]

int_cols = [
    'LCSC_Qty', 'Invoice_Qty', 'Carton_Qty'
]

# 1️⃣ Convert text columns
for col in text_cols:
    if col in EIS.columns:
        EIS[col] = EIS[col].astype(str)

# 2️⃣ Convert date columns
for col in date_cols:
    if col in EIS.columns:
        EIS[col] = pd.to_datetime(EIS[col], errors='coerce')

# 3️⃣ Convert float columns
for col in float_cols:
    if col in EIS.columns:
        EIS[col] = pd.to_numeric(EIS[col], errors='coerce')

# 4️⃣ Convert int columns (use 'Int64' to allow NaN)
for col in int_cols:
    if col in EIS.columns:
        EIS[col] = EIS[col].astype('Int64')

# Optional: check the updated data types
print(EIS.dtypes)


Company                             object
LCSC_No                             object
LCSC_Value                         float64
LCSC_Qty                             Int64
LCSC_Payment_Term                   object
Invoice_No                          object
Invoice_Date                datetime64[ns]
EXP_No                              object
EXP_Date                    datetime64[ns]
Buyer                               object
Merchandiser                        object
Destination                         object
Invoice_Qty                          Int64
Carton_Qty                           Int64
Invoice_Value                      float64
Ex_Factory_Date             datetime64[ns]
Ex_Factory_No                       object
Ex_Factory                          object
On_Board_Date               datetime64[ns]
Payment_Due_Date            datetime64[ns]
On_Board_Diff                      float64
Shipping_Bill_No                    object
Shipping_Bill_Date          datetime64[ns]
BL_No      

In [23]:
EIS['Invoice_Date']

0       2025-09-18
1       2026-02-01
2       2025-08-28
3       2024-09-29
4       2025-03-23
           ...    
43591   2024-06-02
43592   2025-12-15
43593   2023-12-30
43594   2023-12-06
43595   2025-12-24
Name: Invoice_Date, Length: 43596, dtype: datetime64[ns]

In [25]:
EIS.sample(5)

,Company,LCSC_No,LCSC_Value,LCSC_Qty,LCSC_Payment_Term,Invoice_No,Invoice_Date,EXP_No,EXP_Date,Buyer,...,Is_FDBC?,Bank,Shipping_Mode,Status,Realize_No,Realize_Date,Realize_Value,Is_Realize?,Category,Payment_After_Ex_Factory
8725,MGSL,MGSL/Women Shirt S.2,2995246.39,619756,EOM+63,2508599,2025-07-03,00001689-019094-2025,2025-07-03,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-11-25-053,2025-11-04,408.36,Yes,Woven,6.0
11075,MGSL,MGSL/H&M Kids DBBL-S.0,7767885.39,2760068,EOM+63,2414156,2024-09-26,00001689-017879-2024,2024-09-26,H&M,...,Yes,DBBL,Sea,Incentive Pending,DR-12-24-008,2024-12-03,712.53,Yes,Woven,6.0
27243,MGSL,MGSL/PRINCETON SHIRT - 8757 S.08,3204657.14,761705,60 Days,2310872,2023-07-08,042/009322/23,2023-07-08,H&M,...,Yes,ABL,Sea,ExFactory Pending,DR-08-23-015,2023-08-08,1150.00,Yes,Woven,6.0
4471,MGSL,DC HK1095625,77652.00,13998,At Sight,2414552,2024-10-05,00001689-018685-2024,2024-10-05,TMS Fashion (HK) Ltd.,...,Yes,DBBL,Sea,Incentive Pending,DR-11-24-038,2024-11-10,23400.65,Yes,Woven,28.0
28870,MGSL,MGSL/ROBIN OXFORD SHIRT SEASON-9,2038510.93,432934,EOM 63,2401116,2024-01-21,042/001920/24,2024-01-21,H&M,...,Yes,ABL,Sea,Incentive Pending,DR-02-24-032,2024-02-18,247.50,Yes,Woven,6.0


In [30]:
EIS.isna().sum()

Company                        0
LCSC_No                        0
LCSC_Value                     0
LCSC_Qty                       0
LCSC_Payment_Term              0
Invoice_No                     0
Invoice_Date                   0
EXP_No                         0
EXP_Date                       0
Buyer                          0
Merchandiser                   0
Destination                    0
Invoice_Qty                    0
Carton_Qty                  1103
Invoice_Value                  0
Ex_Factory_Date             1212
Ex_Factory_No                  0
Ex_Factory                     0
On_Board_Date                962
Payment_Due_Date             967
On_Board_Diff               2004
Shipping_Bill_No               0
Shipping_Bill_Date          7246
BL_No                          0
BL_Date                     2234
FDBC_No                        0
Bank_Submit_Date            3251
Is_FDBC?                       0
Bank                           0
Shipping_Mode                  0
Status    

In [24]:
STOP

NameError: name 'STOP' is not defined

In [ ]:
import pandas as pd
from datetime import timedelta

# --------------------------
# Convert date columns to datetime
# --------------------------
date_cols = ["Invoice Date", "EXP Date", "Expected Realize Date"]
for col in date_cols:
    EIS[col] = pd.to_datetime(EIS[col], errors='coerce')

# --------------------------
# Set ExfactoryDate (assuming EXP Date)
# --------------------------
EIS['ExfactoryDate'] = EIS['EXP Date']

# --------------------------
# Convert LC/SC Payment Term to days
# --------------------------
def parse_payment_term(term):
    if pd.isna(term):
        return 0
    term = str(term).upper().strip()
    
    if "EOM" in term:
        # EOM + days
        if '+' in term:
            try:
                return int(term.split('+')[1])
            except:
                return 0
        elif ' ' in term:
            try:
                return int(term.split(' ')[-1])
            except:
                return 0
        else:
            return 0
    elif "AT SIGHT" in term:
        return 0
    else:
        # Assume numeric days
        try:
            return int(term.split()[0])
        except:
            return 0

EIS['PaymentAfterExFactory'] = EIS['LC/SC Payment Term'].apply(parse_payment_term)

# --------------------------
# Determine if Bank Ref No is realizable
# --------------------------
def bank_ref_status(row):
    if pd.isna(row['ExfactoryDate']):
        return ""
    due_date = row['ExfactoryDate'] + timedelta(days=row['PaymentAfterExFactory'])
    if pd.Timestamp.today() > due_date:
        return "NeedRealize"
    else:
        return "NotYetRealize"

EIS['IsBankRefRealizable'] = EIS.apply(bank_ref_status, axis=1)

# --------------------------
# Fetch all unique Bank Ref Nos
# --------------------------
unique_bank_refs = EIS['LC/SC No'].dropna().unique()

report_rows = []

for i, bank_ref in enumerate(unique_bank_refs, start=1):
    group = EIS[EIS['LC/SC No'] == bank_ref]
    
    total_invoice = group['Invoice No'].nunique()
    total_invoice_value = group['LC/SC Value'].sum()
    
    min_onboard = group['EXP Date'].min().strftime("%d-%b-%Y") if pd.notna(group['EXP Date'].min()) else ""
    max_onboard = group['EXP Date'].max().strftime("%d-%b-%Y") if pd.notna(group['EXP Date'].max()) else ""
    
    last_invoice_date = group['Invoice Date'].max().strftime("%d-%b-%Y") if pd.notna(group['Invoice Date'].max()) else ""
    
    invoices = ",".join(group['Invoice No'].astype(str))
    
    report_rows.append({
        "SL#": i,
        "Bank Ref No": bank_ref,
        "Is Bank Ref No Realizable?": group['IsBankRefRealizable'].iloc[0],
        "LC Factory": group['LC Factory'].iloc[0],
        "Category": group['Category'].iloc[0],
        "Invoice Status": group['Invoice Status'].iloc[0],
        "Bank Name": group['Bank Name'].iloc[0],
        "Bank Submit Date": group['Expected Realize Date'].iloc[0].strftime("%d-%b-%Y") 
                            if pd.notna(group['Expected Realize Date'].iloc[0]) else "",
        "ExfactoryDate": group['ExfactoryDate'].iloc[0].strftime("%d-%b-%Y") 
                            if pd.notna(group['ExfactoryDate'].iloc[0]) else "",
        "LC/SC Payment Term": group['LC/SC Payment Term'].iloc[0],
        "Payment After Ex Factory": group['PaymentAfterExFactory'].iloc[0],
        "Is Bank Submit?": "YES" if pd.notna(group['Expected Realize Date'].iloc[0]) else "NO",
        "Total Invoice": total_invoice,
        "Total Invoice Value": round(total_invoice_value, 2),
        "On Board Date(Range)": f"{min_onboard} -- {max_onboard}",
        "Last Invoice Date": last_invoice_date,
        "INVOICES": invoices
    })

# --------------------------
# Create final report DataFrame
# --------------------------
report_df = pd.DataFrame(report_rows)

# --------------------------
# Filter only Invoice Status = 'Realize Pending'
# --------------------------
report_df = report_df[report_df['Invoice Status'] == 'Realize Pending']

# Reset index
report_df.reset_index(drop=True, inplace=True)

# --------------------------
# Save to Excel
# --------------------------
report_df.to_excel("Export_Invoice_Report.xlsx", index=False)
print("Report generated successfully!")


In [ ]:
report_df.sample(5)